# ⚛ ⚡️ Quantum formulation of energy network optimization




## Intro
In this notebook I formulate an energy network optimization problem as a fully discrete (binary) optimization problem for solution using a quantum annealer. 

The system is modelled as a set of `Plant`s connected to `Hub`s and the hubs are connected with each other through transmission lines (`Line` objects).

### 📝 Notation and mathematical formulation for single hub:

We denote the set of $n_h$ *hubs* 
 
$H : \{h_{i} | i \in 1,\dots n_h\} $

the set of $n_p$ *plants* 

$P : \{p_{i} | \in 1 ,\cdots n_p\}$

The set of plants connected to hub $h$ will be denoted as $P_h$.

Each plant $p_i$ has a maximum production $k_i$, and a cost of production per unit $c_i$ and is connected to only one hub. 

Each hub will have a demand level $d_h$ which is going to be an input to a problem.

The goal is to find a production level per plant that will exactly satisfy the hub's demand D_h. 
The optimization problem reads:

$
\displaystyle\argmin_{p_i} \displaystyle\sum_{i} c_i \cdot p_i 
$

subsject to the maximum production level (per-plant)

$
p_i \leq c_i \, \forall  i \\
$

and to the hub demand:

$
\displaystyle\sum_{p_i \in P_h} p_i = D_h
$

Note that in the above equation D_h is a scalar. Later we will include further "hubs" and demands which will require us to index them, so we simply introduce the $h$ index early.

> Note: 
> 
> Typically the production costs are taken at least as quadradic for energy system optimization. 
>
> This is not hard to do, but given the additional complexity of mapping the problem to the quantum annealer formuation for the time 
> being I'm only going to treat the linear term.

> Note: 
> 
> After discretizing this problem, it is essentially a Knapsack problem (see [this lecture](youtube.com/watch?v=wFP5VHGHFdk) if you would like to learn more)


#### 📌 Constraints (single hub)

The max production is easy to explicitly satisfy through the maximum representable number for the assigned bits (per-plant). 

This will become clearer below, where the discrete encoding of the numbers is detailed.

The constraint for the demand satisfaction, will be implemented through a penalty term. 

We re-formulate the equality above as the following potential term (that will be added to our problem's objective function)

$
P_{D_h} = \lambda_{D_h} (\displaystyle\sum p_i - D_h )^2
$

The penalty term is going to create a "peak" in the high-dimensional potential for the values that do not satisfy the constraint. 

This form is convenient also due to the "QUBO" formulation that is required for solving with a quantum annealer and we will introduce shortly.


## ✍ Objective function (for a single hub)

The constrained problem now reads:

$
\displaystyle\argmin_{p_i} \displaystyle_i c_i p_i + \lambda_{D_h} (\displaystyle\sum p_i - D_h)^2
$ 


In what follows, we will discretize the all the associated variables ($p_i, c_i, D_h$) for use in a quantum annealer.

----


# ⚛ Problem representation in QUBO form


## Representing numbers in binary form for quantum computations:

In order to model our system, we will need to represent somehow real-world physical quantities. 
The "classical" computing way to do that is with binary numbers.

A positive integer $x_p$ can be represented in N_b bits as 

$ 
x_p = q_{p,0} \cdot 2^0 + q_{p,1} \cdot  2^1 + \dots + q_{p, N_b}2^{N_b}
$

or 

$
x_b = \displaystyle\sum_{i=0}^{N_b} q_{p, i} 2 ^i
$

This is straight-forwardly extendable to positive finite floating point numbers capped at $f_{max}$ as 

$
x_b = \displaystyle\sum_{i=0}^{N_b} q_{p, i} 2 ^i \cdot \frac{f_{max}}{2^{N_b}-1}.
$

The denominator is from the max representable integer with $N_b$ bits which is $2^{N_b} -1 $.

Since positive floating point numbers are enough for representing this problem, we will simply use this representation for our problem.

> **Note:**
>
> An interesting investigation would be to check if there are smarter, more bit-economical ways to encode numbers in a quantum computer. 
> 
> E.g., intuitively, we may want the quantization to be non-uniform in some manner, since two different quantizations may perform differently for the same problem. 
> 
> It's quite interesting to try and discover "that manner"...


### Formulating the problem for solution in a quantum annealer

We want to set up a quantum system in a way that when it slowly evolves to a low energy state, it gives us the solution to our problem.

A canonical form  this is possible for our problem formulation, is by mapping our problem to a **Quadradic Unconstrained Binary Optimization (QUBO)** problem.

The problem formulation reads as follows:


$
Obj(a_i, b_i; q_i) = \displaystyle\sum_{i}a_i q_i + \displaystyle\sum_{i<j} b_{ij} q_i q_j
$

In the formula above,
*  the $q_i$ terms are the "unknowns" which are going to be the outputs of our optimization algorithm (i.e., the *qubits*)
* the $a_i$ are linear terms we can control programmatically. If I understand this correctly, with this term we *bias* a qubit to have a higher probability to be 1 or 0 when we sample it. 
* The coupling terms $b_{ij}$, which correlate our cubits. 

In quantum annealing, we slowly "relax" our system to a low energy state, that will (hopefully!) minimize the term above.

If you are familiar with the simulated annealing technique and MCMC-type methods like the Metropolis algorithm, this may be a familiar setting.

The difference between the classical techniques, is that the quantum annealer exhibits quantum effects, and therefore can escape more easily local minima in the optimization procedure.

**Further learning**
Some words to "google" (or ask ChatGPT if you prefer) to understand how this is possible is "Adiabatic Quantum Computing", and (simulated) annealing. 

I found this video [Quantum Computing Tutorial Part 1: Quantum annealing, QUBOs and more](https://www.youtube.com/watch?v=teraaPiaG8s), 

The rest of D-Wave's videos are also very helpful to understand what's going on in a quantum computer.

----


## Further implementation details


### Encoding `Plants`: 
Each plant has a maximum production capacity, and a cost associated per-unit of production. As mentioned above, I simply constrain the plant's max production by its encoding. 



# 👨‍💻 Code (single hub)

In [7]:
import numpy as np

import numpy as np
from dimod import BinaryQuadraticModel, ExactSolver

# Function to convert QUBO matrix to dimod BQM
def qubo_to_bqm(Q):
    """
    Converts a QUBO matrix to a BinaryQuadraticModel.
    
    :param Q: QUBO matrix as a numpy array.
    :return: BinaryQuadraticModel object.
    """
    # Create a BinaryQuadraticModel
    bqm = BinaryQuadraticModel('BINARY')
    
    # Fill linear terms (diagonal of Q)
    for i in range(Q.shape[0]):
        bqm.add_variable(i, Q[i, i])
    
    # Fill quadratic terms (off-diagonal of Q)
    for i in range(Q.shape[0]):
        for j in range(i + 1, Q.shape[1]):
            if Q[i, j] != 0:
                bqm.add_interaction(i, j, Q[i, j])
    
    return bqm

class EncodedNumber:
    def __init__(self, name, max_value, num_bits, cost_per_unit=0):
        """
        Encodes a number as a binary vector.
        :param name: Name of the variable (for reference).
        :param max_value: Maximum value the number can take.
        :param num_bits: Number of binary bits used to encode the number.
        :param cost_per_unit: Cost per unit of production for this number.
        """
        self.name = name
        self.max_value = max_value
        self.num_bits = num_bits
        _sc = self.max_value / (2**self.num_bits - 1)
        self.weights = np.array([2**i for i in range(num_bits)])* _sc  # Binary weights for each bit
        self.cost_per_unit = cost_per_unit  

class NumberSum:
    def __init__(self, encoded_numbers):
        """
        Represents a sum of encoded numbers, tracking bit offsets for easier handling.
        :param encoded_numbers: List of EncodedNumber objects.
        """
        self.encoded_numbers = encoded_numbers
        self.bit_offsets = self._compute_bit_offsets()

    def _compute_bit_offsets(self):
        """
        Computes the bit offset for each EncodedNumber in the list.
        :return: A dictionary mapping EncodedNumber names to their bit offsets.
        """
        offsets = {}
        current_offset = 0
        for num in self.encoded_numbers:
            offsets[num.name] = current_offset
            current_offset += num.num_bits
        return offsets

    def get_bit_indices(self, encoded_number):
        """
        Gets the global bit indices for an EncodedNumber.
        :param encoded_number: The EncodedNumber object.
        :return: A list of global bit indices.
        """
        start_idx = self.bit_offsets[encoded_number.name]
        return list(range(start_idx, start_idx + encoded_number.num_bits))

    def total_bits(self):
        """
        Computes the total number of bits across all EncodedNumbers.
        :return: Total number of bits.
        """
        return sum(num.num_bits for num in self.encoded_numbers)

    def create_qubo_matrix(self, target_sum, lambda_val=1):
        """
        Constructs the QUBO matrix for the sum of the numbers, enforcing the constraint and considering costs.
        :param target_sum: The desired sum of all numbers.
        :param lambda_val: Penalty parameter.
        :return: QUBO matrix as a numpy array.
        """
        total_bits = self.total_bits()
        Q = np.zeros((total_bits, total_bits))

        # Fill QUBO matrix for each number
        for num in self.encoded_numbers:
            indices = self.get_bit_indices(num)

            # Diagonal and off-diagonal terms within the number
            for i in range(num.num_bits):
                # Add cost term contribution
                Q[indices[i], indices[i]] += num.cost_per_unit * num.weights[i]
                
                # Add constraint penalty contribution
                Q[indices[i], indices[i]] += lambda_val * (num.weights[i] ** 2) - 2 * lambda_val * target_sum * num.weights[i]
                
                for j in range(i + 1, num.num_bits):
                    Q[indices[i], indices[j]] += 2 * lambda_val * num.weights[i] * num.weights[j]
                    Q[indices[j], indices[i]] += 2 * lambda_val * num.weights[i] * num.weights[j]

        # Cross-terms between different numbers
        for i, num1 in enumerate(self.encoded_numbers):
            for j, num2 in enumerate(self.encoded_numbers):
                if i >= j:
                    continue  # Avoid double-counting
                indices1 = self.get_bit_indices(num1)
                indices2 = self.get_bit_indices(num2)
                for k in range(num1.num_bits):
                    for l in range(num2.num_bits):
                        Q[indices1[k], indices2[l]] += 2 * lambda_val * num1.weights[k] * num2.weights[l]
                        Q[indices2[l], indices1[k]] += 2 * lambda_val * num1.weights[k] * num2.weights[l]

        return Q

In [72]:
class Plant:
    def __init__(self, plant_name = 'Plant', max_production = 10, cost_per_unit = 10, num_bits = 5):
        self.max_production = max_production
        self.cost_per_unit = cost_per_unit
        self.num_bits = num_bits
        self.name = plant_name
        self.number_repr = EncodedNumber(plant_name, max_value = max_production, num_bits=num_bits, cost_per_unit=self.cost_per_unit)
        self.is_solved = False
    def set_solution(self, production):
        self.production = production
        self.is_solved = True
    def cost(self):
        if not self.is_solved:
            raise Exception("plant not solved! Cant give cost.")
        
        return self.production * self.cost_per_unit
        
class Hub:
    def __init__(self,  plants, demand):
        self.plants = plants
        self.demand = demand
        self.plants_map  = {p.name : p for p in self.plants}
        self.number_sum = NumberSum([p.number_repr for p in self.plants])
        self.is_solved = False
    def set_solution(self, decoded_solution):
        self.decoded_solution = decoded_solution
        for plant_name, plant_production in self.decoded_solution.items():
            self.plants_map[plant_name].set_solution(plant_production)
        self.is_solved = True
    def cost(self):
        if self.is_solved:
            tot_cost = 0
            for _,  p in self.plants_map.items():
                tot_cost += p.cost()
            return tot_cost
        raise Exception("Hub not solved! Can't return cost.")

In [73]:
class HubOnlyExactSolver:
    """Formulated the QUBO problem and solves it by 
    exhaustive search.
    """
    def __init__(self, hub : Hub):
        self.hub = hub
        
    def solve(self, lambda_val = 10, qubo_solver = None):
        target_sum = self.hub.demand
        vals = self.hub.number_sum
        qubo_matrix = vals.create_qubo_matrix(target_sum, lambda_val)
        bqm = qubo_to_bqm(qubo_matrix)
        if qubo_solver is None:
            solver = ExactSolver()
            
        sampleset = solver.sample(bqm)
            
        # Extract the best solution
        best_solution = sampleset.first.sample  # Binary solution
        best_energy = sampleset.first.energy  # Energy of the best solution
        
        # Decode the solution
        decoded_solution = {}
        number_sum_obj = self.hub.number_sum
        encoded_numbers = self.hub.number_sum.encoded_numbers
        for num in encoded_numbers:
            bits = [best_solution[i] for i in number_sum_obj.get_bit_indices(num)]
            decoded_value = sum(b * w for b, w in zip(bits, num.weights))
            decoded_solution[num.name] = decoded_value
        self.hub.set_solution(decoded_solution)
        res = {
            'Q' : qubo_matrix, 
            'sampleset' : sampleset,
            'best_solution' : best_solution, 
            'best_energy' : best_energy,
            'decoded_solution' : decoded_solution
        }
        
        return decoded_solution
    

## Concrete problem and solution (single hub)

In the following, I define a set of plants with state variables discretized using 5 bits each.

| plant | cost | max production | num_bits   |
|-------|------|----------------|------------|
| Plant 1 | 0.1  | 10           | 5        |
| Plant 2 | 0    | 8            | 5        |
| Plant 3 | 3    | 3            | 5        |

I use these plants to satisfy a demand of 10.5.




In [74]:
plants = [
    Plant('plant 1', max_production=10, cost_per_unit = 0.1, num_bits=1),
    Plant('plant 2', max_production=15, cost_per_unit = 8, num_bits=8),
    Plant('plant 3', max_production=3, cost_per_unit = 1, num_bits=2)
]

hub = Hub(plants=plants, demand=12.0)
hub_solver = HubOnlyExactSolver(hub)
res = hub_solver.solve(lambda_val=1000)
print(res)
print('Gap: ',hub.demand - sum(res.values()))

{'plant 1': 10.0, 'plant 2': 0.0, 'plant 3': 2.0}
Gap:  0.0


> Note:
> 
> It is a good idea to "sweep" the penalty factor. If it is too large the solver may try to satisfy only the constraints it corresponds to 
> while ignoring the minimization objective, and if it is too small the opposite may happen (i.e., the constraints may be ignored)

>  TODO: 
>
>    Extend this to solve also networks with multiple connected hubs to be more realistic.